
# MIE312 Lab 2 — Head Losses in Pipes and Fittings

Developed by Prof. Sullivan (pierre.sullivan@utoronto.ca) If you wish to suggest improvements (and please do make suggestions), please send them to the email address above.

This notebook is set up so you can enter your **measured Lab 2 data** once and then:
- compute Reynolds number, velocity, measured head loss, and theoretical head loss,
- compare **manometer vs DPT** measurements,
- analyze **major losses** in straight pipe sections,
- analyze **minor losses** in fittings,
- plot **head loss vs flow rate**,
- generate short report-ready summary text.

### How to use in Colab
1. Open this notebook from the link in the lab manual and choose **File → Save a copy in Drive**.
2. Run the setup cells from top to bottom.
3. Replace the example values in the data-entry tables with your own measurements.
4. Re-run the calculation and plotting cells.
5. Copy the generated summary text into your lab report and edit as needed.

> **2026 update:** For turbulent pipe flow, the notebook now supports both **Haaland** and **Swamee–Jain** explicit friction-factor correlations. The transitional range is identified separately rather than being assigned a unique turbulent correlation.



## Lab context

The lab handout states that the objective is to **study head losses in various pipes and fittings and compare experimental and theoretical results**.

The apparatus includes multiple horizontal pipe runs, a differential pressure transmitter (DPT), a U-tube manometer, several fittings, and two main pipe diameters used in the analysis. The handout also provides the apparatus dimensions and fitting formulas used in this notebook.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.precision", 5)
pd.set_option("display.max_columns", None)

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.grid"] = True

# Fluid properties (edit if your TA wants a different temperature assumption)
g = 9.81                     # m/s^2
rho = 1000.0                # kg/m^3
mu = 1.00e-3                # Pa*s
nu = mu / rho               # m^2/s

# Smooth PVC pipe roughness from the lab handout
EPS_SMOOTH = 1.5e-6         # m

print("Setup complete.")
print(f"Water density = {rho:.1f} kg/m^3")
print(f"Water viscosity = {mu:.4e} Pa*s")
print(f"Smooth-pipe roughness = {EPS_SMOOTH:.2e} m")

## Geometry and helper functions


### Friction-factor guidance — use this consistently

For **fully developed laminar pipe flow**,
\[
f=\frac{64}{Re}.
\]

For **turbulent pipe flow**, the Colebrook–White equation is the standard implicit relation,
\[
\frac{1}{\sqrt f}
=
-2\log_{10}\left(
\frac{\varepsilon/D}{3.7}
+
\frac{2.51}{Re\sqrt f}
\right),
\]
but it requires iteration because \(f\) appears on both sides.

Two convenient **explicit** alternatives used in MIE312 are:

**Haaland**
\[
\frac{1}{\sqrt f}
\approx
-1.8\log_{10}\left[
\left(\frac{\varepsilon/D}{3.7}\right)^{1.11}
+
\frac{6.9}{Re}
\right],
\]

**Swamee–Jain**
\[
f\approx
\frac{0.25}
{\left[
\log_{10}\left(
\frac{\varepsilon/D}{3.7}
+
\frac{5.74}{Re^{0.9}}
\right)
\right]^2}.
\]

> **Important:** \(2100 \lesssim Re \lesssim 4000\) is the **transitional range**. There is no single turbulent friction-factor correlation that should be treated as uniquely valid there. This notebook therefore reports the regime as transitional and leaves the theoretical friction factor blank rather than silently applying a turbulent correlation.

The main calculations below use the method selected by `FRICTION_FACTOR_METHOD`. The default is **Swamee–Jain**, but you can change it to `"haaland"` and rerun the notebook. For turbulent cases, both should give very similar results; a Moody-chart reading can be used as a graphical cross-check.


In [ ]:

# Key dimensions from the handout (Table 1)
geom = {
    "D_large": 27.2e-3,     # m
    "D_small": 17.0e-3,     # m
    "D_contract_small": 13.6e-3,  # m
    "D_orifice": 18.0e-3,   # m
    "elbow_radius": 1.5e-2, # m
}

segment_meta = {
    "22-23": {"L": 1.00, "D": geom["D_small"], "eps": EPS_SMOOTH},
    "22-25": {"L": 2.00, "D": geom["D_small"], "eps": EPS_SMOOTH},
    "24-25": {"L": 0.50, "D": geom["D_small"], "eps": EPS_SMOOTH},
    "26-28": {"L": 1.00, "D": geom["D_large"], "eps": EPS_SMOOTH},
    "26-29": {"L": 2.00, "D": geom["D_large"], "eps": EPS_SMOOTH},
    "12-13": {"L": 1.00, "D": geom["D_large"], "eps": np.nan},  # rough pipe, estimate later
    "12-15": {"L": 2.00, "D": geom["D_large"], "eps": np.nan},
}

# Choose the explicit turbulent correlation used for the main theoretical columns.
# Options: "swamee-jain" or "haaland"
FRICTION_FACTOR_METHOD = "swamee-jain"


def L_hr_to_m3_s(Q_L_hr):
    return Q_L_hr / 1000 / 3600


def mbar_to_pa(dp_mbar):
    return dp_mbar * 100.0


def mmH2O_to_m(mm_h2o):
    return mm_h2o / 1000.0


def area_circular(D):
    return np.pi * D**2 / 4


def velocity_from_Q(Q_m3_s, D):
    return Q_m3_s / area_circular(D)


def reynolds_number(Q_m3_s, D, rho=rho, mu=mu):
    V = velocity_from_Q(Q_m3_s, D)
    return rho * V * D / mu


def head_loss_from_dpt(dp_mbar, rho=rho, g=g):
    return mbar_to_pa(dp_mbar) / (rho * g)


def head_loss_from_manometer(mm_h2o):
    return mmH2O_to_m(mm_h2o)


def flow_regime(Re):
    if pd.isna(Re):
        return np.nan
    if Re < 2100:
        return "Laminar"
    if Re < 4000:
        return "Transitional"
    return "Turbulent"


def friction_factor_swamee_jain(Re, eps, D):
    """Darcy friction factor from the Swamee–Jain explicit approximation."""
    if pd.isna(Re) or Re <= 0 or pd.isna(eps) or D <= 0:
        return np.nan
    return 0.25 / (np.log10(eps/(3.7*D) + 5.74/(Re**0.9)))**2


def friction_factor_haaland(Re, eps, D):
    """Darcy friction factor from the Haaland explicit approximation."""
    if pd.isna(Re) or Re <= 0 or pd.isna(eps) or D <= 0:
        return np.nan
    inv_sqrt_f = -1.8 * np.log10((eps/(3.7*D))**1.11 + 6.9/Re)
    return 1.0 / (inv_sqrt_f**2)


def friction_factor(Re, eps, D, method=None):
    """
    Darcy friction factor for this notebook.

    Laminar: f = 64/Re.
    Transitional (2100 <= Re < 4000): no unique friction factor is assigned.
    Turbulent (Re >= 4000): use the selected explicit approximation.
    """
    if pd.isna(Re) or Re <= 0:
        return np.nan
    if Re < 2100:
        return 64.0 / Re
    if Re < 4000:
        return np.nan

    method = (method or FRICTION_FACTOR_METHOD).lower().replace("_", "-")
    if method in {"swamee-jain", "swameejain"}:
        return friction_factor_swamee_jain(Re, eps, D)
    if method == "haaland":
        return friction_factor_haaland(Re, eps, D)
    raise ValueError("method must be 'swamee-jain' or 'haaland'")


def major_head_loss(f, L, D, V, g=g):
    return f * (L / D) * (V**2 / (2*g))


def percent_difference(measured, theoretical):
    return 100 * (measured - theoretical) / theoretical


def sudden_contraction_K(d_small, D_large):
    return 0.42 * (1 - (d_small / D_large)**2)


def sudden_expansion_K(d_small, D_large):
    return (1 - (d_small / D_large)**2)**2


def orifice_K(d, D):
    # Gibson correlation from the current lab manual
    return 2.8 * (1 - d/D)**2 * ((D/d)**2 - 1)


def elbow_K_ITO(Re, r, D):
    # Handout formula for 90-degree elbow, turbulent regime
    return 0.367 * (r / D)**(-0.84) + 1.71 * (r / D)**(-1.11) * (Re**(-0.17))



## Section A — Part 1: Manometer vs DPT on the same segment

The handout asks you to compare total head loss measured using:
- the **U-tube manometer**,
- the **differential pressure transmitter (DPT)**,
- and the **theoretical head loss** for the same pipe segment.

For the horizontal constant-diameter section,

$$
h_L = \frac{p_1-p_2}{\rho g}
$$

and for theory on a straight pipe section,

$$
h_{L,major} = f\frac{L}{D}\frac{\bar V^2}{2g}.
$$


In [ ]:

# Replace the example values below with your own Part 1 measurements
part1 = pd.DataFrame({
    "pipe_segment": ["22-23"],
    "Q_L_hr": [2600.0],
    "dh_manometer_mmH2O": [72.0],
    "dP_DPT_mbar": [65.0],
})

part1

In [ ]:

part1_calc = part1.copy()

part1_calc["Q_m3_s"] = part1_calc["Q_L_hr"].apply(L_hr_to_m3_s)
part1_calc["D_m"] = part1_calc["pipe_segment"].map(lambda s: segment_meta[s]["D"])
part1_calc["L_m"] = part1_calc["pipe_segment"].map(lambda s: segment_meta[s]["L"])
part1_calc["eps_m"] = part1_calc["pipe_segment"].map(lambda s: segment_meta[s]["eps"])
part1_calc["V_m_s"] = velocity_from_Q(part1_calc["Q_m3_s"], part1_calc["D_m"])
part1_calc["Re"] = reynolds_number(part1_calc["Q_m3_s"], part1_calc["D_m"])
part1_calc["Regime"] = part1_calc["Re"].apply(flow_regime)
part1_calc["f"] = [friction_factor(Re, eps, D) for Re, eps, D in zip(part1_calc["Re"], part1_calc["eps_m"], part1_calc["D_m"])]

part1_calc["h_measured_manometer_m"] = part1_calc["dh_manometer_mmH2O"].apply(head_loss_from_manometer)
part1_calc["h_measured_DPT_m"] = part1_calc["dP_DPT_mbar"].apply(head_loss_from_dpt)
part1_calc["h_theory_m"] = major_head_loss(part1_calc["f"], part1_calc["L_m"], part1_calc["D_m"], part1_calc["V_m_s"])
part1_calc["pct_diff_DPT_vs_theory"] = percent_difference(part1_calc["h_measured_DPT_m"], part1_calc["h_theory_m"])
part1_calc["pct_diff_manometer_vs_theory"] = percent_difference(part1_calc["h_measured_manometer_m"], part1_calc["h_theory_m"])

part1_calc

# Sanity check: for the same two pressure taps, manometer and DPT head losses should be comparable.
for _, row in part1_calc.iterrows():
    hm = row.get("h_measured_manometer_m", np.nan)
    hd = row.get("h_measured_DPT_m", np.nan)
    if np.isfinite(hm) and np.isfinite(hd) and min(abs(hm), abs(hd)) > 0:
        ratio = max(abs(hm), abs(hd)) / min(abs(hm), abs(hd))
        if ratio > 3:
            print(
                f"UNIT/CHECK WARNING for {row['pipe_segment']}: manometer and DPT head losses differ by "
                f"a factor of {ratio:.1f}. Confirm the manometer units/decimal, DPT units, pressure-tap pairing, "
                "and that both readings were taken at the same flow rate before interpreting the comparison."
            )


In [ ]:

valid = part1_calc.dropna(subset=["h_measured_manometer_m", "h_measured_DPT_m", "h_theory_m"])

if len(valid) > 0:
    plt.figure()
    x = np.arange(len(valid))
    w = 0.25
    plt.bar(x - w, valid["h_measured_manometer_m"], width=w, label="Manometer")
    plt.bar(x, valid["h_measured_DPT_m"], width=w, label="DPT")
    plt.bar(x + w, valid["h_theory_m"], width=w, label="Theory")
    plt.xticks(x, valid["pipe_segment"])
    plt.ylabel("Head loss (m)")
    plt.title("Part 1: Measured vs theoretical head loss")
    plt.legend()
    plt.show()
else:
    print("Enter at least one complete Part 1 row first.")


### Discussion prompts for Part 1
- Why is there a practical flow-rate limit for the manometer?
- Which instrument agreed more closely with theory?
- Are the differences consistent with reading uncertainty, trapped air, calibration drift, or unmodeled losses?


## Section B — Part 2: Major head losses in straight pipe segments

For the straight horizontal runs, the measured head loss comes from the DPT, and the theoretical prediction uses the Darcy–Weisbach equation.

This section is organized for the pipe segments listed in the lab handout:
- **D:** 12-13 ($L=1.00$ m) and 12-15 ($L=2.00$ m), rough pipe,
- **F:** 22-25 ($L=2.00$ m),
- **G:** 26-28 ($L=1.00$ m) and 26-29 ($L=2.00$ m).

You can also compare with **22-23** from Part 1. The smooth-pipe value $\varepsilon=1.5\,\mu$m is an **absolute** surface roughness; relative roughness is $\varepsilon/D$.


### Friction-factor method used here

The theoretical columns use the explicit turbulent correlation selected by `FRICTION_FACTOR_METHOD`.  
The table also reports **both Haaland and Swamee–Jain values side-by-side** for turbulent rows so you can see that the choice of explicit approximation is usually a small effect compared with experimental scatter.


In [ ]:

# Replace with your own Part 2 data
part2 = pd.DataFrame({
    "pipe_segment": ["12-13", "12-15", "22-25", "26-28", "26-29"],
    "Q_L_hr": [2600.0, 2600.0, 2600.0, 2600.0, 2600.0],
    "dP_DPT_mbar": [11.0, 23.0, 119.0, 9.0, 21.0],
})

part2

In [ ]:

part2_calc = part2.copy()
part2_calc["Q_m3_s"] = part2_calc["Q_L_hr"].apply(L_hr_to_m3_s)
part2_calc["D_m"] = part2_calc["pipe_segment"].map(lambda s: segment_meta[s]["D"])
part2_calc["L_m"] = part2_calc["pipe_segment"].map(lambda s: segment_meta[s]["L"])
part2_calc["eps_m"] = part2_calc["pipe_segment"].map(lambda s: segment_meta[s]["eps"])
part2_calc["V_m_s"] = velocity_from_Q(part2_calc["Q_m3_s"], part2_calc["D_m"])
part2_calc["Re"] = reynolds_number(part2_calc["Q_m3_s"], part2_calc["D_m"])
part2_calc["Regime"] = part2_calc["Re"].apply(flow_regime)
part2_calc["h_measured_m"] = part2_calc["dP_DPT_mbar"].apply(head_loss_from_dpt)

# For the rough pipe D, leave the friction-factor columns blank until roughness is estimated below.
part2_calc["f_Haaland"] = [
    friction_factor_haaland(Re, eps, D) if (not pd.isna(eps) and Re >= 4000) else np.nan
    for Re, eps, D in zip(part2_calc["Re"], part2_calc["eps_m"], part2_calc["D_m"])
]
part2_calc["f_Swamee_Jain"] = [
    friction_factor_swamee_jain(Re, eps, D) if (not pd.isna(eps) and Re >= 4000) else np.nan
    for Re, eps, D in zip(part2_calc["Re"], part2_calc["eps_m"], part2_calc["D_m"])
]
part2_calc["f"] = [
    friction_factor(Re, eps, D) if not pd.isna(eps) else np.nan
    for Re, eps, D in zip(part2_calc["Re"], part2_calc["eps_m"], part2_calc["D_m"])
]
part2_calc["f_method"] = [
    ("64/Re" if Re < 2100 else
     "Transitional — no unique correlation" if Re < 4000 else
     FRICTION_FACTOR_METHOD)
    for Re in part2_calc["Re"]
]

part2_calc["h_theory_m"] = [
    major_head_loss(f, L, D, V) if not pd.isna(f) else np.nan
    for f, L, D, V in zip(part2_calc["f"], part2_calc["L_m"], part2_calc["D_m"], part2_calc["V_m_s"])
]
part2_calc["pct_diff_vs_theory"] = percent_difference(part2_calc["h_measured_m"], part2_calc["h_theory_m"])

part2_calc



### Estimate the roughness of pipe D from segment 12-13

The handout asks you to estimate the roughness of pipe D from measured data. We can do this by:
1. computing the measured head loss,
2. solving Darcy–Weisbach for the friction factor,
3. **inverting the Swamee–Jain explicit correlation** to estimate an effective roughness \(\varepsilon\).

This is an **effective roughness inferred from the measured pressure loss**, not a direct physical measurement of the wall texture.

The inversion below is appropriate only when the measured flow is clearly turbulent. If the row falls in the transitional range, do not force a unique turbulent roughness estimate.


In [ ]:

# Roughness estimate using segment 12-13
row = part2_calc.loc[part2_calc["pipe_segment"] == "12-13"].iloc[0]

h = row["h_measured_m"]
L = row["L_m"]
D = row["D_m"]
V = row["V_m_s"]
Re = row["Re"]

f_est = h * D * 2 * g / (L * V**2)

# Invert Swamee–Jain algebraically to estimate roughness:
# f = 0.25 / [log10(eps/(3.7D) + 5.74/Re^0.9)]^2
# => eps = 3.7D [10^(-1/(2sqrt(f))) - 5.74/Re^0.9]
if Re >= 4000:
    term = 10 ** (-1 / (2 * np.sqrt(f_est))) - 5.74 / (Re ** 0.9)
    eps_est = 3.7 * D * term if term > 0 else np.nan
else:
    eps_est = np.nan
    print(
        f"Roughness inversion not performed: Re = {Re:.0f} is not clearly turbulent "
        "(use Re >= 4000 for this Swamee–Jain inversion)."
    )

roughness_estimate = pd.DataFrame({
    "segment_used": ["12-13"],
    "Re": [Re],
    "f_estimated_from_measured_h": [f_est],
    "eps_estimated_m": [eps_est],
    "eps_estimated_um": [eps_est * 1e6 if pd.notna(eps_est) else np.nan],
    "smooth_pipe_eps_um": [EPS_SMOOTH * 1e6],
    "inversion_method": ["Swamee–Jain"],
})

display(roughness_estimate)

# Use the estimated roughness for both rough-pipe D segments so the Part 2
# theoretical comparison and plot are complete.
if pd.notna(eps_est) and eps_est > 0:
    mask_D = part2_calc["pipe_segment"].isin(["12-13", "12-15"])
    part2_calc.loc[mask_D, "eps_m"] = eps_est

    for idx in part2_calc.index[mask_D]:
        Re_i = part2_calc.loc[idx, "Re"]
        D_i = part2_calc.loc[idx, "D_m"]
        L_i = part2_calc.loc[idx, "L_m"]
        V_i = part2_calc.loc[idx, "V_m_s"]

        if Re_i >= 4000:
            f_h = friction_factor_haaland(Re_i, eps_est, D_i)
            f_sj = friction_factor_swamee_jain(Re_i, eps_est, D_i)
        else:
            f_h = np.nan
            f_sj = np.nan

        f_i = friction_factor(Re_i, eps_est, D_i)

        part2_calc.loc[idx, "f_Haaland"] = f_h
        part2_calc.loc[idx, "f_Swamee_Jain"] = f_sj
        part2_calc.loc[idx, "f"] = f_i
        part2_calc.loc[idx, "f_method"] = (
            "64/Re" if Re_i < 2100 else
            "Transitional — no unique correlation" if Re_i < 4000 else
            FRICTION_FACTOR_METHOD
        )

        if pd.notna(f_i):
            part2_calc.loc[idx, "h_theory_m"] = major_head_loss(f_i, L_i, D_i, V_i)
            part2_calc.loc[idx, "pct_diff_vs_theory"] = percent_difference(
                part2_calc.loc[idx, "h_measured_m"],
                part2_calc.loc[idx, "h_theory_m"],
            )

part2_calc


In [ ]:

valid = part2_calc.dropna(subset=["h_measured_m"])

plt.figure()

# Measured data
plt.bar(
    valid["pipe_segment"],
    valid["h_measured_m"],
    color="0.70",
    edgecolor="black",
    label="Measured"
)

# Theoretical prediction
if valid["h_theory_m"].notna().any():
    plt.plot(
        valid["pipe_segment"],
        valid["h_theory_m"],
        color="tab:orange",
        marker="o",
        linewidth=2,
        label="Theory"
    )

plt.ylabel("Head loss (m)")
plt.title("Part 2: Major head losses by pipe segment")
plt.legend()
plt.show()

## Section C — Part 3: Minor losses in fittings

For fittings, the handout gives

$$
h_{L,minor} = K_L \frac{\bar V^2}{2g},
$$

where the velocity should be taken as the **largest mean velocity associated with the fitting**.

The notebook uses the same correlations as the handout, including

$$
K_{L,\mathrm{orifice}}=2.8\left(1-\frac dD\right)^2
$$

and

$$
K_{L,\mathrm{elbow}}=0.367\left(\frac rD\right)^{-0.84}
+1.71\left(\frac rD\right)^{-1.11}Re^{-0.17}.
$$

For segments with a diameter change, the **measured total head loss** includes the kinetic-energy term from the full energy equation:

$$
h_L = \frac{p_1-p_2}{\rho g} + \frac{V_1^2 - V_2^2}{2g} + (z_1-z_2),
$$

and for these horizontal segments, $z_1=z_2$.


In [ ]:

# Replace with your own Part 3 measurements
part3 = pd.DataFrame({
    "pipe_segment": ["6-7", "16-17", "18-19", "16-19", "19-21"],
    "Q_L_hr": [2600.0, 2600.0, 2600.0, 2600.0, 2600.0],
    "dP_DPT_mbar": [76.0, 145.0, -30.0, 131.0, 26.0],
})

part3

In [ ]:

part3_calc = part3.copy()
part3_calc["Q_m3_s"] = part3_calc["Q_L_hr"].apply(L_hr_to_m3_s)
part3_calc["dP_Pa"] = part3_calc["dP_DPT_mbar"].apply(mbar_to_pa)

# Precompute velocities
part3_calc["V_large_m_s"] = velocity_from_Q(part3_calc["Q_m3_s"], geom["D_large"])
part3_calc["V_small_contract_m_s"] = velocity_from_Q(part3_calc["Q_m3_s"], geom["D_contract_small"])
part3_calc["V_orifice_m_s"] = velocity_from_Q(part3_calc["Q_m3_s"], geom["D_orifice"])
part3_calc["Re_large"] = reynolds_number(part3_calc["Q_m3_s"], geom["D_large"])

# Measured total head loss
h_measured = []
K_vals = []
h_minor_theory = []
h_major_theory = []
h_total_theory = []
notes = []

for _, row in part3_calc.iterrows():
    seg = row["pipe_segment"]
    Q = row["Q_m3_s"]
    dP = row["dP_Pa"]
    V_large = row["V_large_m_s"]
    V_small = row["V_small_contract_m_s"]
    V_orifice = row["V_orifice_m_s"]
    Re_large = row["Re_large"]
    base = dP / (rho * g)

    if seg == "6-7":
        # Orifice only, same pipe diameter upstream/downstream
        h_m = base
        K = orifice_K(geom["D_orifice"], geom["D_large"])
        h_minor = K * V_orifice**2 / (2*g)
        h_major = 0.0
        note = "Orifice; same inlet/outlet diameter"

    elif seg == "16-17":
        # sudden contraction; use full Bernoulli kinetic-energy correction
        h_m = base + (V_large**2 - V_small**2) / (2*g)
        K = sudden_contraction_K(geom["D_contract_small"], geom["D_large"])
        h_minor = K * V_small**2 / (2*g)
        h_major = 0.0
        note = "Sudden contraction"

    elif seg == "18-19":
        # sudden expansion
        h_m = base + (V_small**2 - V_large**2) / (2*g)
        K = sudden_expansion_K(geom["D_contract_small"], geom["D_large"])
        h_minor = K * V_small**2 / (2*g)
        h_major = 0.0
        note = "Sudden expansion"

    elif seg == "16-19":
        # contraction + 20 cm small straight + expansion
        h_m = base
        K = sudden_contraction_K(geom["D_contract_small"], geom["D_large"]) + sudden_expansion_K(geom["D_contract_small"], geom["D_large"])
        h_minor = K * V_small**2 / (2*g)
        Re_small = reynolds_number(Q, geom["D_contract_small"])
        f_small = friction_factor(Re_small, EPS_SMOOTH, geom["D_contract_small"])
        h_major = major_head_loss(f_small, 0.20, geom["D_contract_small"], V_small)
        note = "Contraction + short small-diameter straight + expansion"

    elif seg == "19-21":
        # four elbows over 95 cm of large pipe
        h_m = base
        K_single = elbow_K_ITO(Re_large, geom["elbow_radius"], geom["D_large"])
        K = 4 * K_single
        h_minor = K * V_large**2 / (2*g)
        f_large = friction_factor(Re_large, EPS_SMOOTH, geom["D_large"])
        h_major = major_head_loss(f_large, 0.95, geom["D_large"], V_large)
        note = "4 sharp 90° elbows + associated straight section"

    else:
        h_m = np.nan
        K = np.nan
        h_minor = np.nan
        h_major = np.nan
        note = ""

    h_measured.append(h_m)
    K_vals.append(K)
    h_minor_theory.append(h_minor)
    h_major_theory.append(h_major)
    h_total_theory.append(h_minor + h_major if pd.notna(h_minor) and pd.notna(h_major) else np.nan)
    notes.append(note)

part3_calc["h_measured_total_m"] = h_measured
part3_calc["K_total"] = K_vals
part3_calc["h_theory_minor_m"] = h_minor_theory
part3_calc["h_theory_major_m"] = h_major_theory
part3_calc["h_theory_total_m"] = h_total_theory
part3_calc["pct_diff_vs_theory"] = percent_difference(part3_calc["h_measured_total_m"], part3_calc["h_theory_total_m"])
part3_calc["notes"] = notes

part3_calc

In [ ]:

valid = part3_calc.dropna(subset=["h_measured_total_m", "h_theory_total_m"])

if len(valid) > 0:
    plt.figure()
    x = np.arange(len(valid))
    w = 0.35
    plt.bar(x - w/2, valid["h_measured_total_m"], width=w, label="Measured total")
    plt.bar(x + w/2, valid["h_theory_total_m"], width=w, label="Theoretical total")
    plt.xticks(x, valid["pipe_segment"])
    plt.ylabel("Head loss (m)")
    plt.title("Part 3: Measured vs theoretical fitting losses")
    plt.legend()
    plt.show()
else:
    print("Enter at least one complete Part 3 row first.")


## Section D — Part 4: Increasing flow rate for one segment

The handout asks you to choose a pipe segment (excluding elbows), record measurements over a range of flow rates, and plot

\[
h_L \text{ vs } Q.
\]

For a fixed pipe geometry,
\[
h_L=f\frac{L}{D}\frac{V^2}{2g},
\qquad
V\propto Q.
\]

If \(f\) were constant, the result would be exactly quadratic: \(h_L\propto Q^2\).  
In turbulent pipe flow, however, \(f\) generally changes with Reynolds number. Over a flow-rate sweep the fitted exponent can therefore be **somewhat below 2**.

The code below reports \(Re\) and the selected explicit friction factor at every point and fits both:
- a quadratic curve, and
- a power law \(h_L=aQ^n\).

Interpret the fitted exponent physically; do **not** describe this Darcy–Weisbach trend as simply “exponential.”


In [ ]:

# Replace the example values below with your own Part 4 measurements
# Default example uses segment 22-25, matching the sample report.
part4 = pd.DataFrame({
    "pipe_segment": ["22-25"] * 6,
    "Q_L_hr": [500, 1000, 2000, 3000, 4000, 5000],
    "dP_DPT_mbar": [7, 20, 76, 155, 265, 393],
})

part4

In [ ]:

part4_calc = part4.copy()
part4_calc["Q_m3_s"] = part4_calc["Q_L_hr"].apply(L_hr_to_m3_s)
part4_calc["h_measured_m"] = part4_calc["dP_DPT_mbar"].apply(head_loss_from_dpt)
part4_calc["Q_squared"] = part4_calc["Q_m3_s"]**2

# Add Reynolds number and friction-factor information for physical interpretation.
part4_calc["D_m"] = part4_calc["pipe_segment"].map(lambda s: segment_meta[s]["D"])
part4_calc["L_m"] = part4_calc["pipe_segment"].map(lambda s: segment_meta[s]["L"])
part4_calc["eps_m"] = part4_calc["pipe_segment"].map(lambda s: segment_meta[s]["eps"])
part4_calc["V_m_s"] = velocity_from_Q(part4_calc["Q_m3_s"], part4_calc["D_m"])
part4_calc["Re"] = reynolds_number(part4_calc["Q_m3_s"], part4_calc["D_m"])
part4_calc["Regime"] = part4_calc["Re"].apply(flow_regime)
part4_calc["f"] = [
    friction_factor(Re, eps, D) if not pd.isna(eps) else np.nan
    for Re, eps, D in zip(part4_calc["Re"], part4_calc["eps_m"], part4_calc["D_m"])
]
part4_calc["h_theory_m"] = [
    major_head_loss(f, L, D, V) if not pd.isna(f) else np.nan
    for f, L, D, V in zip(
        part4_calc["f"],
        part4_calc["L_m"],
        part4_calc["D_m"],
        part4_calc["V_m_s"],
    )
]

# Fit a quadratic in Q and a power law h = a Q^n for interpretation.
coeffs = np.polyfit(part4_calc["Q_m3_s"], part4_calc["h_measured_m"], 2)
poly = np.poly1d(coeffs)

# Power-law fit requires positive Q and positive head loss.
fit_mask = (part4_calc["Q_m3_s"] > 0) & (part4_calc["h_measured_m"] > 0)
logQ = np.log(part4_calc.loc[fit_mask, "Q_m3_s"])
logh = np.log(part4_calc.loc[fit_mask, "h_measured_m"])
power_n, loga = np.polyfit(logQ, logh, 1)
power_a = np.exp(loga)

part4_calc


In [ ]:

Q_line = np.linspace(part4_calc["Q_m3_s"].min(), part4_calc["Q_m3_s"].max(), 200)

plt.figure()
plt.scatter(part4_calc["Q_m3_s"], part4_calc["h_measured_m"], s=60, label="Measured data")
plt.plot(Q_line, poly(Q_line), linewidth=2, label="Quadratic fit")
plt.xlabel("Flow rate, Q (m$^3$/s)")
plt.ylabel("Head loss, h$_L$ (m)")
plt.title("Part 4: Head loss vs flow rate")
plt.legend()
plt.show()

print("Quadratic fit:")
print(f"h_L(Q) = {coeffs[0]:.3e} Q^2 + {coeffs[1]:.3e} Q + {coeffs[2]:.3e}")
print()
print("Power-law fit:")
print(f"h_L(Q) = {power_a:.3e} * Q^{power_n:.3f}")


## Section E — Report-ready summary

Run the next cell after entering your actual measurements. It will print a concise discussion-style summary that you can adapt for your report.


In [ ]:
summary_lines = []

# Part 1 summary
p1 = part1_calc.dropna(subset=["h_measured_DPT_m", "h_theory_m"])
if len(p1) > 0:
    row = p1.iloc[0]
    summary_lines.append("## Part 1 — Manometer vs DPT")
    summary_lines.append(
        f"For segment {row['pipe_segment']}, the DPT-based head loss was {row['h_measured_DPT_m']:.3f} m while the manometer-based head loss was {row['h_measured_manometer_m']:.3f} m."
    )
    summary_lines.append(
        f"The theoretical Darcy–Weisbach prediction was {row['h_theory_m']:.3f} m, giving percent differences of {row['pct_diff_DPT_vs_theory']:.2f}% for the DPT and {row['pct_diff_manometer_vs_theory']:.2f}% for the manometer."
    )

# Part 2 summary
p2 = part2_calc.dropna(subset=["h_measured_m"])
if len(p2) > 0:
    max_seg = p2.loc[p2['h_measured_m'].idxmax(), 'pipe_segment']
    min_seg = p2.loc[p2['h_measured_m'].idxmin(), 'pipe_segment']
    summary_lines.append("## Part 2 — Major losses")
    summary_lines.append(
        f"Among the measured straight segments, {max_seg} had the largest measured head loss and {min_seg} had the smallest."
    )
    if roughness_estimate["eps_estimated_um"].notna().all():
        eps_um = roughness_estimate.loc[0, 'eps_estimated_um']
        summary_lines.append(
            f"Using segment 12-13, the estimated roughness of pipe D was approximately {eps_um:.1f} μm."
        )

# Part 3 summary
p3 = part3_calc.dropna(subset=["h_measured_total_m", "h_theory_total_m"])
if len(p3) > 0:
    worst = p3.iloc[np.argmax(np.abs(p3['pct_diff_vs_theory'].values))]
    summary_lines.append("## Part 3 — Minor losses")
    summary_lines.append(
        f"The fitting segment with the largest absolute deviation from theory was {worst['pipe_segment']}, with a percent difference of {worst['pct_diff_vs_theory']:.2f}%."
    )
    summary_lines.append(
        "This is consistent with the fact that local losses depend strongly on flow separation, mixing, and geometry-specific effects that are difficult to model perfectly with handbook coefficients."
    )

# Part 4 summary
if len(part4_calc) > 1:
    summary_lines.append("## Part 4 — Flow-rate trend")
    summary_lines.append(
        f"The power-law fit gave h_L ∝ Q^{power_n:.3f}. A value below 2 is physically reasonable "
        "when the Darcy friction factor decreases as Reynolds number increases across the sweep."
    )
    finite_re = part4_calc["Re"].dropna()
    finite_f = part4_calc["f"].dropna()
    if len(finite_re) > 0:
        summary_lines.append(
            f"Across these points, Re ranged from approximately {finite_re.min():.2e} to {finite_re.max():.2e}."
        )
    if len(finite_f) > 0:
        summary_lines.append(
            f"Using {FRICTION_FACTOR_METHOD}, the Darcy friction factor ranged from approximately "
            f"{finite_f.max():.4f} at the lower-Re end to {finite_f.min():.4f} at the higher-Re end."
        )

if summary_lines:
    display(Markdown("\n\n".join(summary_lines)))
else:
    print("Enter your measurements first, then rerun this cell.")



## Section F — Submission checklist

Before submitting, consider adding:
- captions under each figure,
- one short uncertainty paragraph,
- one short conclusion paragraph,
- a clear statement of the **friction-factor method** used for turbulent flow: Moody chart, Colebrook iteration, Haaland, or Swamee–Jain,
- a note that the \(2100<Re<4000\) range is transitional and does not have a unique turbulent friction-factor prescription,
- any TA-specific formatting requirements.

For a turbulent calculation, Haaland and Swamee–Jain should normally give similar friction factors. If your result differs strongly from a Moody-chart reading, re-check \(Re\), \(\varepsilon/D\), units, and whether you are using the **Darcy** rather than Fanning friction factor.

This notebook is designed to save time on repetitive calculations so you can focus on interpretation and discussion. You are still responsible for understanding the equations, checking the inputs, and showing any worked example required by the lab instructions.
